Python code to generat a sample data for e-commerce product recommendation system.
This program will generate two csv files.


1.   User interactions table (e.g. Web and mobile clickstream logs) having 100000 records
2.   Product table (Product metadata from catalogs) having 100 records
2.   User table (User master table) having 5000 records







In [2]:
!pip install faker

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from faker import Faker

# Initialize Faker
fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

# ==========================================
# CONFIGURATION
# ==========================================
NUM_PRODUCTS = 100
NUM_USERS = 5000
NUM_INTERACTIONS = 100000
ERROR_RATE = 0.05  # 5% error rate

# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================

def inject_noise(df, rate=0.05):
    """
    Injects ~5% errors: Missing Values, Duplicates, and Schema Mismatches.
    """
    df_noisy = df.copy()
    n_rows, n_cols = df_noisy.shape

    # --- A. Inject Random Missing Values (NaN) ---
    total_cells = n_rows * n_cols
    n_missing = int(total_cells * rate * 0.5)

    for _ in range(n_missing):
        row_idx = random.randint(0, n_rows - 1)
        col_idx = random.randint(0, n_cols - 1)
        df_noisy.iat[row_idx, col_idx] = np.nan

    # --- B. Inject Schema Mismatch (Type Errors) ---
    n_mismatch = int(n_rows * rate * 0.25)
    # Target columns that aren't IDs to avoid breaking joins completely
    target_cols = [c for c in df.columns if 'id' not in c]

    if target_cols:
        for _ in range(n_mismatch):
            row_idx = random.randint(0, n_rows - 1)
            col_name = random.choice(target_cols)
            df_noisy.at[row_idx, col_name] = "Invalid_Data"

    # --- C. Inject Duplicate Entries ---
    n_dupes = int(n_rows * rate * 0.25)
    dupes = df_noisy.sample(n=n_dupes, replace=True)
    df_noisy = pd.concat([df_noisy, dupes], ignore_index=True)

    # Shuffle
    df_noisy = df_noisy.sample(frac=1).reset_index(drop=True)

    return df_noisy

# ==========================================
# 2. GENERATE DATASETS
# ==========================================

# --- A. Product Table ---
print("Generating Product Table...")
categories = ['Electronics', 'Fashion', 'Home', 'Beauty', 'Sports']
products = []

for i in range(1, NUM_PRODUCTS + 1):
    cat = random.choice(categories)
    # Generating a fake product name (e.g., "Small Rubber Keyboard")
    p_name = f"{fake.word().title()} {fake.word().title()} {random.choice(['Pro', 'Max', 'Lite', 'v2'])}"

    products.append({
        'product_id': f'P{i:03d}',
        'product_name': p_name,           # Added
        'category': cat,
        'brand': fake.company(),
        'price': round(random.uniform(10, 1500), 2),
        'rating': round(random.uniform(1.0, 5.0), 1), # Added
        'stock_level': random.randint(0, 500)
    })

df_products_clean = pd.DataFrame(products)
df_products_dirty = inject_noise(df_products_clean, ERROR_RATE)

# --- B. User Table ---
print("Generating User Table...")
users = []

for i in range(1, NUM_USERS + 1):
    users.append({
        'user_id': f'U{i:05d}',
        'name': fake.name(),
        'email': fake.email(),
        'gender': random.choice(['M', 'F', 'Other']),
        'age': random.randint(18, 70),
        'city': fake.city(),
        'signup_date': fake.date_between(start_date='-2y', end_date='today')
    })

df_users_clean = pd.DataFrame(users)
df_users_dirty = inject_noise(df_users_clean, ERROR_RATE)

# --- C. User Interactions Table ---
print("Generating User Interactions Table...")
valid_user_ids = [x['user_id'] for x in users]
valid_product_ids = [x['product_id'] for x in products]
actions = ['view', 'click', 'add_to_cart', 'purchase']

interactions = []

# Generate timestamps
start_time = datetime.now() - timedelta(days=90)
timestamps = [start_time + timedelta(seconds=random.randint(0, 90*24*3600)) for _ in range(NUM_INTERACTIONS)]
timestamps.sort()

for i in range(NUM_INTERACTIONS):
    interactions.append({
        'interaction_id': f'INT{i:08d}',
        'user_id': random.choice(valid_user_ids),
        'product_id': random.choice(valid_product_ids),
        'action': random.choices(actions, weights=[60, 25, 10, 5])[0],
        'rating': round(random.uniform(1.0, 5.0), 1) if random.random() < 0.1 else None, # 10% chance of rating
        'timestamp': timestamps[i],
        'device': random.choice(['Mobile', 'Desktop', 'Tablet']),
        'session_duration_sec': random.randint(5, 1800) # Added (5 sec to 30 mins)
    })

df_interactions_clean = pd.DataFrame(interactions)
df_interactions_dirty = inject_noise(df_interactions_clean, ERROR_RATE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 4.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Generating Product Table...
Generating User Table...


/tmp/ipykernel_34997/1519294806.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Invalid_Data' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_noisy.at[row_idx, col_name] = "Invalid_Data"


Generating User Interactions Table...


/tmp/ipykernel_34997/1519294806.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Invalid_Data' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_noisy.at[row_idx, col_name] = "Invalid_Data"
/tmp/ipykernel_34997/1519294806.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Invalid_Data' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_noisy.at[row_idx, col_name] = "Invalid_Data"
/tmp/ipykernel_34997/1519294806.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Invalid_Data' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_noisy.at[row_idx, col_name] = "Invalid_Data"


In [ ]:
# from google.colab import drive
import os

# --- Step 4: Save Directly to Google Drive Folder ---
# print("Uploading to Google Drive...")

# Mount GoSave to the mounted drive path
# drive.mount('/content/drive')
# Save to the mounted drive path
# df_products_dirty.to_csv('/content/drive/My Drive/Sem-2/DMML/DMML-Group-4/products_dirty.csv', index=False)
df_products_clean.to_csv('/home/abhishek/Documents/Study/dmml_assignment/group_4_dmml/data/source_raw/products_dirty.csv', index=False)
df_users_dirty.to_csv('/home/abhishek/Documents/Study/dmml_assignment/group_4_dmml/data/source_raw/users_dirty.csv', index=False)
df_interactions_dirty.to_csv('/home/abhishek/Documents/Study/dmml_assignment/group_4_dmml/data/source_raw/interactions_dirty.csv', index=False)
print("CSV saved successfully.")

# df_products_dirty.to_csv('products_dirty.csv', index=False)
# df_users_dirty.to_csv('users_dirty.csv', index=False)
# df_interactions_dirty.to_csv('interactions_dirty.csv', index=False)


# # The folder ID extracted from your link:
# folder_id = 'https://drive.google.com/drive/folders/1kq1cWSVU3LJgyo92zYuhJNzy-UYvQ6jl'
# print(f"Success! Files have been saved to folder ID: {folder_id}")

Uploading to Google Drive...
CSV saved successfully.
Success! Files have been saved to folder ID: https://drive.google.com/drive/folders/1kq1cWSVU3LJgyo92zYuhJNzy-UYvQ6jl


In [ ]:
# !pip install faker

# import pandas as pd
# import numpy as np
# import random
# from datetime import datetime, timedelta
# from faker import Faker

# # Initialize Faker
# fake = Faker()
# Faker.seed(42)
# random.seed(42)
# np.random.seed(42)

# # ==========================================
# # CONFIGURATION
# # ==========================================
# NUM_PRODUCTS = 100
# NUM_USERS = 5000
# NUM_INTERACTIONS = 100000
# ERROR_RATE = 0.05  # 5% error rate

# # ==========================================
# # 1. HELPER FUNCTIONS
# # ==========================================

# def inject_noise(df, rate=0.05):
#     """
#     Injects ~5% errors: Missing Values, Duplicates, and Schema Mismatches.
#     """
#     df_noisy = df.copy()
#     n_rows, n_cols = df_noisy.shape

#     # --- A. Inject Random Missing Values (NaN) ---
#     # Calculate total cells to blank out based on rate
#     total_cells = n_rows * n_cols
#     n_missing = int(total_cells * rate * 0.5) # Allocating half the error budget to missing values

#     for _ in range(n_missing):
#         row_idx = random.randint(0, n_rows - 1)
#         col_idx = random.randint(0, n_cols - 1)
#         df_noisy.iat[row_idx, col_idx] = np.nan

#     # --- B. Inject Schema Mismatch (Type Errors) ---
#     # Example: Putting a string in a numeric column or a bad date
#     n_mismatch = int(n_rows * rate * 0.25) # 25% of error budget
#     target_cols = [c for c in df.columns if 'id' not in c] # Avoid breaking foreign keys completely if possible

#     if target_cols:
#         for _ in range(n_mismatch):
#             row_idx = random.randint(0, n_rows - 1)
#             col_name = random.choice(target_cols)
#             # Insert a string garbage value
#             df_noisy.at[row_idx, col_name] = "Invalid_Data"

#     # --- C. Inject Duplicate Entries ---
#     # We append duplicates to the end
#     n_dupes = int(n_rows * rate * 0.25) # Remaining 25% of error budget
#     dupes = df_noisy.sample(n=n_dupes, replace=True)
#     df_noisy = pd.concat([df_noisy, dupes], ignore_index=True)

#     # Shuffle the dataset to mix duplicates and originals
#     df_noisy = df_noisy.sample(frac=1).reset_index(drop=True)

#     return df_noisy

# # ==========================================
# # 2. GENERATE DATASETS
# # ==========================================

# # --- A. Product Table (Metadata) ---
# print("Generating Product Table...")
# categories = ['Electronics', 'Fashion', 'Home', 'Beauty', 'Sports']
# products = []

# for i in range(1, NUM_PRODUCTS + 1):
#     cat = random.choice(categories)
#     products.append({
#         'product_id': f'P{i:03d}',
#         'category': cat,
#         'brand': fake.company(),
#         'price': round(random.uniform(10, 500), 2),
#         'stock_level': random.randint(0, 100)
#     })

# df_products_clean = pd.DataFrame(products)
# df_products_dirty = inject_noise(df_products_clean, ERROR_RATE)

# # --- B. User Table (Demographics) ---
# print("Generating User Table...")
# users = []

# for i in range(1, NUM_USERS + 1):
#     users.append({
#         'user_id': f'U{i:05d}',
#         'name': fake.name(),
#         'email': fake.email(),
#         'gender': random.choice(['M', 'F', 'Other']),
#         'age': random.randint(18, 70),
#         'city': fake.city(),
#         'signup_date': fake.date_between(start_date='-2y', end_date='today')
#     })

# df_users_clean = pd.DataFrame(users)
# df_users_dirty = inject_noise(df_users_clean, ERROR_RATE)

# # --- C. User Interactions Table (Clickstream) ---
# print("Generating User Interactions Table...")
# # Note: We must use the CLEAN IDs to ensure initial foreign key integrity
# # before we inject noise.
# valid_user_ids = [x['user_id'] for x in users]
# valid_product_ids = [x['product_id'] for x in products]
# actions = ['view', 'click', 'add_to_cart', 'purchase']

# interactions = []

# # Generate timestamps
# start_time = datetime.now() - timedelta(days=90)
# timestamps = [start_time + timedelta(seconds=random.randint(0, 90*24*3600)) for _ in range(NUM_INTERACTIONS)]
# timestamps.sort() # Sort to simulate chronological logs

# for i in range(NUM_INTERACTIONS):
#     interactions.append({
#         'interaction_id': f'INT{i:08d}',
#         'user_id': random.choice(valid_user_ids),
#         'product_id': random.choice(valid_product_ids),
#         'action': random.choices(actions, weights=[60, 25, 10, 5])[0],
#         'timestamp': timestamps[i],
#         'device': random.choice(['Mobile', 'Desktop', 'Tablet'])
#     })

# df_interactions_clean = pd.DataFrame(interactions)
# df_interactions_dirty = inject_noise(df_interactions_clean, ERROR_RATE)

# # ==========================================
# # 3. SUMMARY & EXPORT
# # ==========================================

# print("\n--- Data Generation Complete ---")
# print(f"Products: {df_products_dirty.shape} (Original target: 100)")
# print(f"Users: {df_users_dirty.shape} (Original target: 5000)")
# print(f"Interactions: {df_interactions_dirty.shape} (Original target: 100000)")

# # Verify Error Injection (Preview)
# print("\n--- Sample of 'Dirty' Data (Products) ---")
# print(df_products_dirty.head())

# print("\n--- Check for Nulls (Interactions) ---")
# print(df_interactions_dirty.isnull().sum())

# # Export to CSV
# # df_products_dirty.to_csv('products_dirty.csv', index=False)
# # df_users_dirty.to_csv('users_dirty.csv', index=False)
# # df_interactions_dirty.to_csv('interactions_dirty.csv', index=False)

# # print("\nFiles saved: products_dirty.csv, users_dirty.csv, interactions_dirty.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.0 MB/s eta 0:00:00
Generating Product Table...
Generating User Table...


/tmp/ipython-input-538275710.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Invalid_Data' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_noisy.at[row_idx, col_name] = "Invalid_Data"


Generating User Interactions Table...


/tmp/ipython-input-538275710.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Invalid_Data' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_noisy.at[row_idx, col_name] = "Invalid_Data"



--- Data Generation Complete ---
Products: (101, 5) (Original target: 100)
Users: (5062, 7) (Original target: 5000)
Interactions: (101250, 6) (Original target: 100000)

--- Sample of 'Dirty' Data (Products) ---
  product_id     category                    brand   price  stock_level
0       P031         Home                      NaN  114.19         63.0
1       P068  Electronics           Novak and Sons  122.18          4.0
2       P063         Home          Wright and Sons  224.67         66.0
3       P048      Fashion  Martin, Rose and Obrien  258.76         13.0
4       P043  Electronics            Rodriguez LLC  344.04         68.0

--- Check for Nulls (Interactions) ---
interaction_id    2480
user_id           2415
product_id        2608
action            2460
timestamp         2428
device            2541
dtype: int64


In [ ]:
# # --- Step 1: Install and Authenticate Google Drive Access ---
# # This is required to save files directly to your Drive folder
# import pandas as pd
# import random
# from datetime import datetime, timedelta
# from pydrive.auth import GoogleAuth
# from pydrive.drive import GoogleDrive
# from google.colab import auth
# from oauth2client.client import GoogleCredentials

# from google.colab import drive
# import os

# # --- Step 2: Generate Product Data (Same logic as before) ---
# print("Generating Product Table...")
# NUM_PRODUCTS = 100
# categories = ['Electronics', 'Clothing', 'Home & Garden', 'Books', 'Beauty', 'Toys']
# product_types = {
#     'Electronics': ['Smartphone', 'Laptop', 'Headphones', 'Smart Watch', 'Tablet'],
#     'Clothing': ['T-Shirt', 'Jeans', 'Jacket', 'Sneakers', 'Dress'],
#     'Home & Garden': ['Lamp', 'Chair', 'Plant Pot', 'Rug', 'Blender'],
#     'Books': ['Novel', 'Biography', 'Textbook', 'Cookbook', 'Comic'],
#     'Beauty': ['Lipstick', 'Perfume', 'Face Cream', 'Shampoo', 'Serum'],
#     'Toys': ['Action Figure', 'Puzzle', 'Doll', 'Board Game', 'Building Blocks']
# }

# products = []
# for i in range(1, NUM_PRODUCTS + 1):
#     cat = random.choice(categories)
#     p_name = f"{random.choice(product_types[cat])} Model-{random.randint(100, 999)}"
#     products.append({
#         'product_id': f'P{i:03d}',
#         'product_name': p_name,
#         'category': cat,
#         'price': round(random.uniform(10.0, 1000.0), 2),
#         'rating': round(random.uniform(1.0, 5.0), 1),
#         'in_stock': random.choice([True, True, True, False])
#     })
# df_products = pd.DataFrame(products)

# # --- Step 3: Generate User Interactions Data ---
# print("Generating User Interactions Table...")
# NUM_INTERACTIONS = 100000
# NUM_USERS = 5000
# actions = ['view', 'click', 'add_to_cart', 'purchase', 'like']
# devices = ['mobile_app', 'mobile_web', 'desktop_web', 'tablet']
# user_ids = [f'U{i:05d}' for i in range(1, NUM_USERS + 1)]
# product_ids = [p['product_id'] for p in products]

# def random_date(start, end):
#     delta = end - start
#     int_delta = (delta.days * 24 * 60 * 60) + delta.seconds
#     random_second = random.randrange(int_delta)
#     return start + timedelta(seconds=random_second)

# d1 = datetime.strptime('1/1/2024 1:30 PM', '%m/%d/%Y %I:%M %p')
# d2 = datetime.now()

# interactions = []
# for i in range(NUM_INTERACTIONS):
#     interactions.append({
#         'interaction_id': f'INT{i:06d}',
#         'user_id': random.choice(user_ids),
#         'product_id': random.choice(product_ids),
#         'timestamp': random_date(d1, d2).isoformat(),
#         'action': random.choices(actions, weights=[50, 30, 10, 5, 5])[0],
#         'device': random.choice(devices),
#         'session_duration_sec': random.randint(1, 300)
#     })
# df_interactions = pd.DataFrame(interactions)

# # --- Step 4: Save Directly to Google Drive Folder ---
# print("Uploading to Google Drive...")

# # Mount GoSave to the mounted drive path
# drive.mount('/content/drive')
# # Save to the mounted drive path
# df_products.to_csv('/content/drive/My Drive/Sem-2/DMML/DMML-Group-4/products_metadata.csv', index=False)
# df_interactions.to_csv('/content/drive/My Drive/Sem-2/DMML/DMML-Group-4/user_interactions.csv', index=False)
# print("CSV saved successfully.")


# # # The folder ID extracted from your link:
# folder_id = 'https://drive.google.com/drive/folders/1kq1cWSVU3LJgyo92zYuhJNzy-UYvQ6jl'
# print(f"Success! Files have been saved to folder ID: https://drive.google.com/drive/folders/1kq1cWSVU3LJgyo92zYuhJNzy-UYvQ6jl")

Generating Product Table...
Generating User Interactions Table...
Uploading to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSV saved successfully.
Success! Files have been saved to folder ID: https://drive.google.com/drive/folders/1kq1cWSVU3LJgyo92zYuhJNzy-UYvQ6jl
